In [ ]:
import sys, os, pickle, warnings
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats as _stats

sys.path.insert(0, '/Users/yotameviatar/startle-1')
warnings.filterwarnings('ignore')

from Airflow import airflow_config as config
from Airflow import airflow_glm as glm
from IPython import get_ipython


# inline backend, set last so it's robust regardless of import order

get_ipython().run_line_magic('matplotlib', 'inline')
plt.rcParams.update({'figure.dpi': 110})
print('Setup complete — backend:', matplotlib.get_backend())

## Setup — cache & GLM scoring (run once)


In [ ]:
import hashlib, json as _json
# ── What to show ──────────────────────────────────────────────────────────────
SUBJECT   = 'MS18'    # subjects - 'AB22', 'AG05', 'AH19', 'AK12', 'AS09', 'DA01'
SESSION   = None      # 'eve', 'mor', or None for both

# ── Layout ───────────────────────────────────────────────────────────────────
NCOLS     = 6
SAVE_FIGS = False     

# ── Group section (bottom of notebook) ────────────────────────────────────────
GROUP_METRIC              = 'ra'    # which metric's per-subject scatter to show
SHOW_PER_SUBJECT_SCATTER  = True   # True = one figure per subject (24 x N figures)

# ── Diagnostics section ───────────────────────────────────────────────────────
SHOW_DIAGNOSTICS = False   # True = run Diagnostics 1/2/3 (condition-N imbalance,
                           # outlier trials, label-scheme comparison). Hidden by
                           # default while not actively debugging labeling/outliers.
FORCE_RESCORE = globals().get("FORCE_RESCORE", False)

cache_path = os.path.join(config.OUTPUT_DIR, "_cache", "airflow_cache.pkl")

with open(cache_path, "rb") as f:
    sessions_cache = pickle.load(f)["sessions"]

def _config_fingerprint():
    """Hash every JSON-able config value + the cache mtime, so ANY change to a
    scoring parameter (or the underlying cache) invalidates the scoring cache."""
    items = {}
    for k in dir(config):
        if k.startswith("_"):
            continue
        v = getattr(config, k)
        if callable(v):
            continue
        try:
            _json.dumps(v)
            items[k] = v
        except TypeError:
            pass
    items["__cache_mtime"] = os.path.getmtime(cache_path)
    blob = _json.dumps(items, sort_keys=True, default=str)
    return hashlib.md5(blob.encode()).hexdigest()[:12]

_score_cache = os.path.join(config.OUTPUT_DIR, "_cache",
                            f"glm_scoring_{_config_fingerprint()}.pkl")

if os.path.exists(_score_cache) and not FORCE_RESCORE:
    with open(_score_cache, "rb") as f:
        _sc = pickle.load(f)
    trials_data, traces = _sc["trials_data"], _sc["traces"]
    print(f"Loaded cached scoring: {os.path.basename(_score_cache)}  "
          f"(FORCE_RESCORE = True to redo)")
else:
    print("Scoring all sessions (session-wide NK2 pass; first time only, then cached) ...")
    trials_data, traces = glm.run_glm_scoring(sessions_cache, config)
    with open(_score_cache, "wb") as f:
        pickle.dump({"trials_data": trials_data, "traces": traces}, f)
    print(f"Scored and cached: {os.path.basename(_score_cache)}")

if SUBJECT not in sessions_cache:
    raise ValueError(f"Subject '{SUBJECT}' not found. Available: {sorted(sessions_cache.keys())}")

if SUBJECT not in trials_data:
    # SUBJECTS_EXCLUDE (airflow_config.py) parks this subject out of ALL
    # scoring -- trials_data/traces never get an entry for it. That's a
    # Layer-2 scoring decision, not a reason signal-only views (0, 0b, 2,
    # the PsPM view) should be unable to load it -- they read sessions_cache
    # directly and don't need trials_data at all.
    print(f"{SUBJECT} is parked in SUBJECTS_EXCLUDE -- not scored (see airflow_config.py). "
          f"Views needing trial-level data (1, 3, ...) will skip it; signal-only views "
          f"(0, 0b, 2, PsPM view) still work.")
    sess_keys = [SESSION] if SESSION else sorted(sessions_cache[SUBJECT].keys())
else:
    sess_keys = ([SESSION] if SESSION else
                 sorted(k for k in trials_data[SUBJECT] if trials_data[SUBJECT][k]))
print(f"{len(trials_data)} subjects scored · {SUBJECT} sessions: {sess_keys}")


### View — PsPM cycle detection + filtering (3 consecutive accepted trials)


In [ ]:
# black dotted = D105 (fixation), purple dotted = image onset, green dotted = sound (D110)
import random
from scipy.signal import resample
from matplotlib.lines import Line2D

PSPM_VIEW_SEED   = None   # int to pin the random pick; None = fresh each run
PSPM_VIEW_METRIC = "RP"   # bottom-panel metric: "RP", "RA", or "RFR"
assert PSPM_VIEW_METRIC in ("RP", "RA", "RFR"), "PSPM_VIEW_METRIC must be RP, RA, or RFR"
METRIC = PSPM_VIEW_METRIC

if SUBJECT not in sessions_cache:
    print(f"{SUBJECT} not in cache (excluded?) — nothing to show.")
else:
    # ── session-wide compute, memoised (mirrors run_glm_scoring) ────────────────
    _sess_memo = {}
    def _session_pspm(sk):
        if sk in _sess_memo:
            return _sess_memo[sk]
        sess = sessions_cache[SUBJECT][sk]
        signal_raw   = sess["signal_raw"]
        native_sfreq = float(sess["sfreq"])
        raw_z, sfreq = glm.phase1_filter_downsample(
            signal_raw, native_sfreq, target_sfreq=config.GLM_TARGET_SFREQ,
            bp_low=config.GLM_CYCLE_BANDPASS[0], bp_high=config.GLM_CYCLE_BANDPASS[1],
            zscore=config.GLM_ZSCORE_RAW,
            despike=config.GLM_DESPIKE_ENABLED, despike_k=config.GLM_DESPIKE_K,
            despike_max_run_sec=config.GLM_DESPIKE_MAX_RUN_SEC)
        n = len(raw_z)
        xr = np.asarray(signal_raw, dtype=np.float64)
        raw_disp = resample(xr - np.nanmean(xr), n)     # RAW (black): downsample + z, NO bandpass
        med = float(np.nanmedian(raw_disp))
        mad = float(np.nanmedian(np.abs(raw_disp - med))) * 1.4826
        if mad > 0:
            raw_disp = (raw_disp - med) / mad
        cycles = glm.detect_cycles(
            raw_z, sfreq, signal_raw, native_sfreq,
            median_win_sec=config.GLM_MEDIAN_WIN_SEC, refractory_sec=config.GLM_REFRACTORY_SEC)
        series, _ = glm.build_continuous_series(
            cycles, n, sfreq, hp=config.GLM_FINAL_HP, lp=config.GLM_FINAL_LP)
        _sess_memo[sk] = dict(raw_disp=raw_disp, raw_z=raw_z, sfreq=sfreq, cycles=cycles,
                              times_full=np.arange(n) / sfreq, m_filt=series[METRIC])
        return _sess_memo[sk]

    _legend = [Line2D([0], [0], color="k", lw=1, label="Raw trace"),
               Line2D([0], [0], color="b", lw=1, label="Filtered trace"),
               Line2D([0], [0], color="k", ls=":", lw=1.5, label="fixation onset (D105)"),
               Line2D([0], [0], color="purple", ls=":", lw=1.5, label="image onset (code)"),
               Line2D([0], [0], color="green", ls=":", lw=1.4, label="sound (D110)")]

    def _annotate_rp_ra(axA, S, cyc_disp, ref, ytop):
        """RP (black horizontal, FIRST cycle) + RA (black vertical) + RFR = RA/RP,
        as in the PsPM figure."""
        if not cyc_disp:
            return
        c0 = cyc_disp[0]                                   # first cycle
        x0, x1 = c0["onset_time"] - ref, c0["assign_time"] - ref
        axA.annotate("", xy=(x1, ytop), xytext=(x0, ytop),
                     arrowprops=dict(arrowstyle="<->", color="k", lw=1.4))
        axA.text((x0 + x1) / 2, ytop, "RP", color="k", ha="center", va="bottom", fontsize=10)
        cra = cyc_disp[min(1, len(cyc_disp) - 1)]          # RA on this cycle (top-to-bottom)
        xr = cra["assign_time"] - ref
        j0 = int(round(cra["onset_time"] * S["sfreq"]))
        j1 = int(round(cra["assign_time"] * S["sfreq"]))
        segr = S["raw_disp"][j0:j1 + 1]
        if len(segr):
            lo, hi = float(np.min(segr)), float(np.max(segr))
            axA.annotate("", xy=(xr, hi), xytext=(xr, lo),
                         arrowprops=dict(arrowstyle="<->", color="k", lw=1.4))
            axA.text(xr, (lo + hi) / 2, " RA", color="k", ha="left", va="center", fontsize=10)
        axA.text(0.98, 0.04, "RFR = RA/RP", transform=axA.transAxes,
                 ha="right", va="bottom", fontsize=9)

    # ── choose ONE run of 3 consecutive accepted trials (shown separately + united)
    rng = random.Random(PSPM_VIEW_SEED)
    runs = []
    for sk in sess_keys:
        tr = trials_data[SUBJECT].get(sk, [])
        for i in range(len(tr) - 2):
            win3 = tr[i:i + 3]
            if all(x.get("rejected") is False for x in win3):
                runs.append((sk, win3))
    if runs:
        sk3, trio = rng.choice(runs)
        picks = [(sk3, t) for t in trio]
        print(f"{SUBJECT}: consecutive trials "
              f"T{', T'.join(str(t['trial_index']) for t in trio)} "
              f"({config.SESSION_MAP.get(sk3, {}).get('label', sk3)}) — separately, then united.")
    else:
        accepted = [(sk, t) for sk in sess_keys
                    for t in trials_data[SUBJECT].get(sk, []) if t.get("rejected") is False]
        sk3, trio = None, None
        picks = rng.sample(accepted, min(3, len(accepted)))
        print(f"{SUBJECT}: no 3 consecutive accepted trials — {len(picks)} random shown; united skipped.")

    # ── (1) each trial SEPARATELY ───────────────────────────────────────────────
    for sk, t in picks:
        S  = _session_pspm(sk)
        sf = S["sfreq"]; code = t["onset_time"]
        i0, i1 = t["win_start_idx"], t["win_end_idx"]
        ref = i0 / sf                                     # t=0 at D105 (fixation onset, window start) -- NOT the image
        tt = S["times_full"][i0:i1] - ref
        cyc = [c for c in S["cycles"] if tt[0] <= (c["onset_time"] - ref) <= tt[-1]]
        both = np.concatenate([S["raw_disp"][i0:i1], S["raw_z"][i0:i1]])
        wmin, wmax = float(np.nanmin(both)), float(np.nanmax(both)); rng_ = wmax - wmin
        slabel = config.SESSION_MAP.get(sk, {}).get("label", sk)

        fig, (axA, axC) = plt.subplots(2, 1, figsize=(9, 5.2), sharex=True,
                                       gridspec_kw=dict(height_ratios=[2, 1], hspace=0.16))
        fig.suptitle(f"{SUBJECT} · {slabel} · trial {t['trial_index']}", fontsize=11)
        axA.plot(tt, S["raw_disp"][i0:i1], color="k", lw=0.8)
        axA.plot(tt, S["raw_z"][i0:i1],    color="b", lw=1.2)
        for c in cyc:                                              # red cycle separators
            axA.axvline(c["onset_time"] - ref, color="r", lw=0.8)
        axA.axvline(0, color="k", ls=":", lw=1.6)                 # fixation onset (D105) at t=0
        axA.axvline(code - ref, color="purple", ls=":", lw=1.6)   # image onset (picture-identity code)
        if t.get("d110_rel") is not None:
            axA.axvline((code - ref) + t["d110_rel"], color="green", ls=":", lw=1.4)   # sound (D110)
        _annotate_rp_ra(axA, S, cyc, ref, wmax + 0.10 * rng_)
        axA.set_ylim(wmin - 0.10 * rng_, wmax + 0.26 * rng_)
        axA.set_ylabel("Transducer output"); axA.set_title("Cycle detection")
        axA.legend(handles=_legend, loc="lower left", fontsize=6.5, framealpha=0.6, ncol=2)

        axC.plot(tt, S["m_filt"][i0:i1], color="k", lw=1.0)
        axC.set_ylabel(f"{METRIC} (s)"); axC.set_title("Filtering"); axC.set_xlabel("Time (s)")
        plt.show()

    # ── (2) the SAME 3 consecutive trials UNITED ────────────────────────────────
    if trio is not None:
        S  = _session_pspm(sk3); sf = S["sfreq"]
        i0, i1 = trio[0]["win_start_idx"], trio[2]["win_end_idx"]
        ref = i0 / sf                                     # t=0 at first trial's D105 (fixation onset) -- NOT the image
        tt = S["times_full"][i0:i1] - ref
        cyc = [c for c in S["cycles"] if tt[0] <= (c["onset_time"] - ref) <= tt[-1]]
        slabel = config.SESSION_MAP.get(sk3, {}).get("label", sk3)
        tids = ", T".join(str(x["trial_index"]) for x in trio)

        fig, (axA, axC) = plt.subplots(2, 1, figsize=(11, 5.7), sharex=True,
                                       gridspec_kw=dict(height_ratios=[2, 1], hspace=0.16))
        fig.suptitle(f"{SUBJECT} · {slabel} · trials T{tids}", fontsize=11)
        axA.plot(tt, S["raw_disp"][i0:i1], color="k", lw=0.8)
        axA.plot(tt, S["raw_z"][i0:i1],    color="b", lw=1.1)
        for c in cyc:                                              # red cycle separators
            axA.axvline(c["onset_time"] - ref, color="r", lw=0.7, alpha=0.85)
        axC.plot(tt, S["m_filt"][i0:i1], color="k", lw=1.0)
        for x in trio:
            axA.axvline(x["win_start_idx"] / sf - ref, color="k", ls=":", lw=1.6)   # fixation onset (D105)
            axA.axvline(x["onset_time"] - ref, color="purple", ls=":", lw=1.6)       # image onset (code)
            if x.get("d110_rel") is not None:                     # sound (D110) on BOTH panels
                xs = (x["onset_time"] - ref) + x["d110_rel"]
                for ax in (axA, axC):
                    ax.axvline(xs, color="green", ls=":", lw=1.4)
        axA.set_ylabel("Transducer output"); axA.set_title("Cycle detection")
        axA.legend(handles=_legend, loc="lower left", fontsize=6.5, framealpha=0.6, ncol=2)
        axC.set_ylabel(f"{METRIC} (s)"); axC.set_title("Filtering"); axC.set_xlabel("Time (s)")
        plt.show()


## View 0b — before/after Butterworth


In [ ]:
FILTER_DEMO_SUBJECT = SUBJECT       # change to plot a different subject
FILTER_DEMO_SESSION = sess_keys[0]  # 'eve' or 'mor'
FILTER_DEMO_MODE    = 'first'       # 'first' = first 2 trials, 'random' = 2 random trials

_sess = sessions_cache[FILTER_DEMO_SUBJECT][FILTER_DEMO_SESSION]
_signal_raw, _native_sfreq = _sess['signal_raw'], float(_sess['sfreq'])

# before = raw signal_raw, mean-centered; after = raw_z (Phase 1 filtered)
_x_before = _signal_raw - np.nanmean(_signal_raw)
_raw_z, _sfreq = glm.phase1_filter_downsample(
    _signal_raw, _native_sfreq,
    target_sfreq=config.GLM_TARGET_SFREQ,
    bp_low=config.GLM_CYCLE_BANDPASS[0],
    bp_high=config.GLM_CYCLE_BANDPASS[1],
    zscore=config.GLM_ZSCORE_RAW,
    despike=config.GLM_DESPIKE_ENABLED,
    despike_k=config.GLM_DESPIKE_K,
    despike_max_run_sec=config.GLM_DESPIKE_MAX_RUN_SEC,
)

_meta = _sess['trials_meta']
if FILTER_DEMO_MODE == 'random':
    _picked = sorted(np.random.choice(len(_meta), size=2, replace=False))
else:
    _picked = [0, 1]

_segments = []
for _i in _picked:
    _m = _meta[_i]
    _t0, _t1 = _m['baseline_range_sec'][0], _m['response_range_sec'][1]
    _i0n, _i1n = int(_t0 * _native_sfreq), int(_t1 * _native_sfreq)
    _i0z, _i1z = int(_t0 * _sfreq), int(_t1 * _sfreq)
    _t_before = np.arange(_i0n, _i1n) / _native_sfreq - _m['code_time']
    _t_after  = np.arange(_i0z, _i1z) / _sfreq - _m['code_time']
    _segments.append((_m, _t_before, _t_after,
                       _x_before[_i0n:_i1n], _raw_z[_i0z:_i1z]))

# Shared x-range across both trials so panels line up in time. Y-axes are
# NOT shared: "before" (signal_raw) is in native flow units, "after" (raw_z)
# is z-scored (SD units) -- comparing shape, not amplitude, across a twin axis.
_xmin = min(t.min() for _, t, _, _, _ in _segments)
_xmax = max(t.max() for _, t, _, _, _ in _segments)

fig, axes = plt.subplots(len(_segments), 1, figsize=(11, 4 * len(_segments)), sharex=True)
if len(_segments) == 1:
    axes = [axes]
for ax, (m, t_before, t_after, before, after) in zip(axes, _segments):
    ax2 = ax.twinx()
    l1, = ax.plot(t_before, before, color='black', linewidth=1.0, alpha=0.7,
                   label='before Phase 1 (signal_raw, mean-centered, native units)')
    l2, = ax2.plot(t_after, after, color='#C0392B', linewidth=1.3,
                    label=f'after Phase 1 ({config.GLM_CYCLE_BANDPASS[0]}-{config.GLM_CYCLE_BANDPASS[1]} Hz '
                          f'cascaded filter + downsample + z-score)')
    ax.axvline(0, color='gray', linestyle='--', linewidth=1, label='trial onset (code_time)')
    ax.set_xlim(_xmin, _xmax)
    ax.set_title(f"{FILTER_DEMO_SUBJECT} {FILTER_DEMO_SESSION} — trial {m['trial_index']} "
                 f"({m['image_type']}, code_value={m['code_value']})")
    ax.set_ylabel('signal_raw (native units)')
    ax2.set_ylabel('raw_z (z-units)')
    ax.legend(handles=[l1, l2], loc='upper right', fontsize=8)
axes[-1].set_xlabel('time relative to trial onset (s)')
plt.tight_layout()
plt.show()
plt.close(fig)


## Rejection Reason Key (GLM path)

| # | Code | What it measures |
|---|---|---|
| 1 | `no_cycles_found` | No breath detected (custom detector + NK2 both disagree) |
| 2 | `noisy_baseline` | Baseline std z-score > `GLM_Z_SCORE_THRESHOLD` |
| 3 | `flat_signal` | Baseline std < `GLM_MIN_STD_RATIO` × session median |
| 4 | `rate_artifact` | Fastest cycle rate > `GLM_RATE_ARTIFACT_THRESHOLD` bpm |
| 5 | `flat_response` | Response std < `GLM_POST_MIN_STD_RATIO` × session median |
| 6 | `atypical_shape` | Breath-shape MSD outlier (`AIRFLOW_SHAPE_SD_THRESHOLD`) |
| 7 | `cycle_gap` | Longest inter-cycle gap implies rate < `GLM_MIN_RATE_THRESHOLD` bpm |
| 8 | `amplitude_artifact` | Hampel-flagged spike (only under `hampel_*` methods) |


### View 1 — Individual Trials (raw_z, GLM path)


In [ ]:
label_name = {1: 'Neg', 2: 'Neu'}
label_col  = {1: config.NEG_COLOR, 2: config.NEU_COLOR}

REJ_LABELS = {
    'no_cycles_found': '#1 No cycles found',
    'noisy_baseline':  '#2 Noisy baseline',
    'flat_signal':     '#3 Flat signal',
    'flat_response':   '#4 Flat response',
    'rate_artifact':   '#5 Rate artifact',
    'cycle_gap':       '#6 Cycle gap',
    'atypical_shape':  '#7 Shape outlier',
    'score_ceiling':   '#8 Score ceiling',
}

def _fmt_reason(rr):
    parts = [REJ_LABELS.get(r.strip(), r.strip()) for r in rr.split(',') if r.strip()]
    return '\n'.join(parts)

def _fmt_score(v):
    return f'{v:.2f}' if v is not None and np.isfinite(float(v)) else '—'

for sess_key in sess_keys:
    trials = trials_data[SUBJECT].get(sess_key, [])
    if not trials:
        continue
    sess_label = config.SESSION_MAP.get(sess_key, {}).get('label', sess_key)

    accepted_vals = [t['epoch_anal'] for t in trials
                     if not t.get('rejected') and t.get('epoch_anal') is not None]
    if accepted_vals:
        flat = np.concatenate(accepted_vals)
        ylo, yhi = np.percentile(flat, 2), np.percentile(flat, 98)
        margin = max((yhi - ylo) * 0.15, np.abs(yhi) * 0.05)
        ylim = (ylo - margin, yhi + margin)
    else:
        ylim = (-3, 3)

    n     = len(trials)
    nrows = max(1, (n + NCOLS - 1) // NCOLS)
    fig, axes = plt.subplots(nrows, NCOLS, figsize=(NCOLS * 3.0, nrows * 2.4), squeeze=False)
    axes_flat = axes.flatten()

    for i, t in enumerate(trials):
        ax  = axes_flat[i]
        tw  = t['times_wide'].astype(float)
        ep  = t.get('epoch_anal')
        rej = t.get('rejected')
        lbl = t.get('label', 0)
        rr  = t.get('rejection_reason', '')

        col       = '#95A5A6' if rej else label_col.get(lbl, '#95A5A6')
        title_col = 'dimgray' if rej else 'black'

        if ep is not None:
            ax.plot(tw, ep.astype(float), color=col, lw=1.0, zorder=3)

        ax.axvspan(tw[0], 0.0, alpha=0.10, color='#3498DB', zorder=0)   # baseline
        ax.axvspan(0.0, tw[-1], alpha=0.08, color='green', zorder=0)    # response
        ax.axvline(0, color='black', lw=0.8, ls='--', alpha=0.45)
        if t.get('has_sound') and t.get('d110_rel') is not None:
            ax.axvline(t['d110_rel'], color='red', lw=1.0, alpha=0.7, zorder=4)

        ax.set_xlim(tw[0], tw[-1])
        ax.set_ylim(ylim)
        ax.tick_params(labelsize=5.5)
        ax.spines[['top', 'right']].set_visible(False)

        if rej:
            title = f"T{i+1} REJ\n{_fmt_reason(rr) or '?'}"
        else:
            title = (f"T{i+1} {label_name.get(lbl,'?')}\n"
                     f"RP={_fmt_score(t.get('score_rp'))} RA={_fmt_score(t.get('score_ra'))} "
                     f"RFR={_fmt_score(t.get('score_rfr'))}")
        ax.set_title(title, fontsize=5.8, color=title_col, pad=2)

    for ax in axes_flat[n:]:
        ax.set_visible(False)

    fig.suptitle(f'{SUBJECT} — {sess_label} — Individual trials (GLM path, N={n})',
                 fontsize=11, fontweight='bold')
    fig.legend(handles=[
        plt.Line2D([0], [0], color=config.NEG_COLOR, lw=1.5, label='Negative'),
        plt.Line2D([0], [0], color=config.NEU_COLOR, lw=1.5, label='Neutral'),
        plt.Line2D([0], [0], color='#95A5A6', lw=1.5, label='Rejected'),
        mpatches.Patch(color='#3498DB', alpha=0.5, label='Baseline window'),
        mpatches.Patch(color='green', alpha=0.3, label='Response window'),
        plt.Line2D([0], [0], color='red', lw=1.5, label='Sound (D110)'),
    ], fontsize=7, loc='lower center', ncol=5, framealpha=0.9, bbox_to_anchor=(0.5, -0.01))

    plt.tight_layout(rect=[0, 0.03, 1, 0.97])

    if SAVE_FIGS:
        outdir = os.path.join(config.OUTPUT_DIR, 'glm_show', SUBJECT)
        os.makedirs(outdir, exist_ok=True)
        fig.savefig(os.path.join(outdir, f'{SUBJECT}_{sess_key}_trials.png'),
                    dpi=config.PLOT_DPI, bbox_inches='tight')
    plt.show()
    plt.close(fig)


### View 2 — Full Session Overview


In [ ]:
excluded_note = ''
if SUBJECT not in trials_data:
    # SUBJECTS_EXCLUDE (airflow_config.py) parks this subject/session out of
    # ALL scoring -- run_glm_scoring() never adds it to trials_data/traces at
    # all, so trials_data[SUBJECT] raises KeyError here rather than just
    # coming back empty. But View 2 is a signal-inspection tool, and the
    # whole point of parking a subject there is "not yet confirmed against
    # the raw signal" (see SUBJECTS_EXCLUDE comments) -- so inspection must
    # still work. Recompute raw_z directly from sessions_cache (same call as
    # View 0b/PsPM view), same despiking, just no trial markers/rejection
    # counts since none were ever scored for this subject.
    excluded_note = '  [EXCLUDED from scoring — see SUBJECTS_EXCLUDE — showing signal only]'
    print(f'{SUBJECT} not in trials_data (parked in SUBJECTS_EXCLUDE) -- '
          f'showing raw_z directly from sessions_cache, no trial markers.')

for sess_key in sess_keys:
    scored_n = None
    scored_start_sec = 0.0
    if SUBJECT in trials_data:
        trials = trials_data[SUBJECT].get(sess_key, [])
        tr = traces.get(SUBJECT, {}).get(sess_key)
        if tr is None or tr.get('signal') is None:
            print(f'{sess_key}: no session trace — skipping')
            continue
        sig   = tr['signal'].astype(float)
        sfreq = tr['sfreq']
        # Crop the view to what the GLM actually scored: GLM_PRE_FIXATION_SEC
        # before the first trial's own fixation onset (d105_time) through
        # GLM_POST_CODE_SEC after the last trial's picture-code onset
        # (code_time). traces[...]['scored_n_samples'/'scored_start_sec']
        # carry both bounds. Showing more would show signal the fit never saw.
        scored_n = tr.get('scored_n_samples', len(sig))
        scored_start_sec = tr.get('scored_start_sec') or 0.0
        scored_start_idx = int(round(scored_start_sec * sfreq))
        sig = sig[scored_start_idx:scored_n]
    else:
        sess = sessions_cache.get(SUBJECT, {}).get(sess_key)
        if sess is None:
            print(f'{sess_key}: not in sessions_cache — skipping')
            continue
        trials = []
        sig, sfreq = glm.phase1_filter_downsample(
            sess['signal_raw'], float(sess['sfreq']),
            target_sfreq=config.GLM_TARGET_SFREQ,
            bp_low=config.GLM_CYCLE_BANDPASS[0], bp_high=config.GLM_CYCLE_BANDPASS[1],
            zscore=config.GLM_ZSCORE_RAW,
            despike=config.GLM_DESPIKE_ENABLED, despike_k=config.GLM_DESPIKE_K,
            despike_max_run_sec=config.GLM_DESPIKE_MAX_RUN_SEC)
        sig = sig.astype(float)

    sess_label = config.SESSION_MAP.get(sess_key, {}).get('label', sess_key)
    t_axis = scored_start_sec + np.arange(len(sig)) / sfreq

    fig, ax = plt.subplots(figsize=(20, 3.5))
    ax.plot(t_axis, sig, color='#2C3E50', lw=0.6, zorder=2, label='raw_z (Phase 1, scored span only)')

    # View 2 is for visual session inspection, not amplitude analysis -- cap
    # the axis at +-20 SD so one outlier trial can't crush every real breath
    # in the session down to a flat line. Scoring/rejection gates elsewhere
    # still see the full-resolution sig; this only clips what's DRAWN.
    y_lo, y_hi = np.nanpercentile(sig, [1, 99])
    margin = max((y_hi - y_lo) * 0.2, 1.0)
    ax.set_ylim(max(y_lo - margin, -20), min(y_hi + margin, 20))

    for t in trials:
        col = '#27AE60' if not t.get('rejected') else '#E74C3C'
        ax.axvline(t['onset_time'], color=col, lw=0.9, alpha=0.75, zorder=4)

    ax.set_xlabel('Time in session (s)', fontsize=9)
    ax.set_ylabel('Airflow (z-scored)', fontsize=9)
    ax.set_xlim(scored_start_sec, scored_start_sec + len(sig) / sfreq)
    n_rej = sum(1 for t in trials if t.get('rejected'))
    title_counts = f'(green=pass, red=reject  {n_rej}/{len(trials)} rejected)' if trials else '(not scored)'
    crop_note = (f'  [cropped to scored span, {scored_start_sec:.0f}-{scored_start_sec + len(sig)/sfreq:.0f}s]'
                 if scored_n is not None else '')
    ax.set_title(
        f'{SUBJECT} — {sess_label} — Full session (raw_z, Phase 1){crop_note}  '
        f'{title_counts}{excluded_note}',
        fontsize=10, fontweight='bold')
    ax.legend(fontsize=8, loc='upper right')
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()

    if SAVE_FIGS:
        outdir = os.path.join(config.OUTPUT_DIR, 'glm_show', SUBJECT)
        os.makedirs(outdir, exist_ok=True)
        fig.savefig(os.path.join(outdir, f'{SUBJECT}_{sess_key}_session.png'),
                    dpi=config.PLOT_DPI, bbox_inches='tight')
    plt.show()
    plt.close(fig)

### View 2b — Baseline Windows Stitched Across the Session


In [ ]:
# baseline segment = sig[win_start_idx : win_start_idx + split], split at times_wide==0
BASELINE_STITCH_INCLUDE_REJECTED = False  # True = also plot rejected trials (grey)
BASELINE_STITCH_MAX_XTICKS       = 40     # thin trial-number ticks if N exceeds this

for sess_key in sess_keys:
    trials = trials_data[SUBJECT].get(sess_key, [])
    if not trials:
        continue
    sess_label = config.SESSION_MAP.get(sess_key, {}).get('label', sess_key)

    tr = traces.get(SUBJECT, {}).get(sess_key)
    if tr is None or tr.get('signal') is None:
        print(f'{sess_key}: no session trace — skipping')
        continue
    sig = tr['signal'].astype(float)

    segments = []
    for t in trials:
        if not BASELINE_STITCH_INCLUDE_REJECTED and t.get('rejected'):
            continue
        tw = t.get('times_wide')
        i0 = t.get('win_start_idx')
        if tw is None or i0 is None or len(tw) == 0:
            continue
        split = int(np.searchsorted(tw, 0.0))  # baseline = times_wide < 0
        if split <= 1:
            continue
        segments.append({
            'trial_index':   t['trial_index'],
            'signal':        sig[i0:i0 + split],
            'rejected':      bool(t.get('rejected')),
            'label':         t.get('label', 0),
        })

    if not segments:
        print(f'{sess_key}: no baseline segments — skipping')
        continue

    fig, ax = plt.subplots(figsize=(20, 3.5))
    x0 = 0
    boundaries, tick_x, tick_lbl = [], [], []
    trend_x, trend_y = [], []
    for seg in segments:
        n = len(seg['signal'])
        x = np.arange(x0, x0 + n)
        col = '#95A5A6' if seg['rejected'] else label_col.get(seg['label'], '#2C3E50')
        ax.plot(x, seg['signal'], color=col, lw=0.6, zorder=2)
        boundaries.append(x0)
        tick_x.append(x0 + n / 2)
        tick_lbl.append(str(seg['trial_index'] + 1))
        trend_x.append(x0 + n / 2)
        trend_y.append(float(np.mean(seg['signal'])))
        x0 += n

    for b in boundaries[1:]:
        ax.axvline(b, color='black', lw=0.4, alpha=0.25, zorder=1)

    ax.plot(trend_x, trend_y, color='#E67E22', lw=1.6, marker='o', ms=3,
            zorder=4, label='per-trial baseline mean')

    # Thin x-ticks to trial numbers -- one per segment if few, else evenly spaced.
    step = max(1, len(tick_x) // BASELINE_STITCH_MAX_XTICKS)
    ax.set_xticks(tick_x[::step])
    ax.set_xticklabels(tick_lbl[::step], fontsize=6.5)

    ax.set_xlabel(f'baseline_1 → baseline_{len(segments)}  (concatenated, '
                  f'response windows removed — NOT real session time)', fontsize=9)
    ax.set_ylabel('Airflow (z-scored)', fontsize=9)
    ax.set_title(
        f'{SUBJECT} — {sess_label} — Baseline windows stitched across session '
        f'(N={len(segments)} trials'
        f'{"" if BASELINE_STITCH_INCLUDE_REJECTED else ", accepted only"})',
        fontsize=10, fontweight='bold')
    ax.legend(fontsize=8, loc='upper right')
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_xlim(0, x0)
    plt.tight_layout()

    if SAVE_FIGS:
        outdir = os.path.join(config.OUTPUT_DIR, 'glm_show', SUBJECT)
        os.makedirs(outdir, exist_ok=True)
        fig.savefig(os.path.join(outdir, f'{SUBJECT}_{sess_key}_baseline_stitch.png'),
                    dpi=config.PLOT_DPI, bbox_inches='tight')
    plt.show()
    plt.close(fig)


### View 2c — Sound vs No-Sound, count-matched panels (reaction contrast)


In [ ]:
# count-matched: N = accepted no-sound trials; sound panels sampled early/mid/late, same N
SOUND_SPLIT_INCLUDE_REJECTED = False   # True = also include rejected trials (grey)
SOUND_MATCH_PANELS           = 3       # how many N-trial sound panels to show

from matplotlib.lines import Line2D

def _draw_run(ax, panel_trials, sfreq, title, mark_sound):
    """Lay a list of trials end-to-end on ax; return mean (resp-base) reaction."""
    x0, reactions = 0.0, []
    for k, t in enumerate(panel_trials):
        ep = np.asarray(t['epoch_anal'], dtype=float)
        tw = np.asarray(t['times_wide'], dtype=float)
        n  = len(ep)
        x  = x0 + np.arange(n) / sfreq
        col = ('#95A5A6' if t.get('rejected')
               else label_col.get(t.get('label', 0), '#2C3E50'))
        ax.plot(x, ep, color=col, lw=0.7, zorder=2)

        onset_off = -float(tw[0])                       # s into window where t=0
        ax.axvline(x0 + onset_off, color='k', ls=':', lw=1.0, alpha=0.7, zorder=3)
        if mark_sound and t.get('d110_rel') is not None:
            ax.axvline(x0 + onset_off + float(t['d110_rel']),
                       color='green', ls=':', lw=1.1, zorder=3)

        base, resp = ep[tw < 0], ep[tw >= 0]
        if base.size and resp.size:
            reactions.append(float(np.mean(resp) - np.mean(base)))

        x0 += n / sfreq
        if k < len(panel_trials) - 1:
            ax.axvline(x0, color='black', lw=0.6, alpha=0.30, zorder=1)  # separator

    mean_react = float(np.mean(reactions)) if reactions else float('nan')
    ax.set_ylabel('Airflow (z)', fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_title(title, fontsize=9.5, fontweight='bold', loc='left')
    ax.text(0.995, 0.90, f'reaction = {mean_react:+.3f} z', ha='right', va='top',
            transform=ax.transAxes, fontsize=8.5,
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#BDC3C7', alpha=0.85))
    return mean_react

for sess_key in sess_keys:
    trials = trials_data[SUBJECT].get(sess_key, [])
    if not trials:
        continue
    sess_label = config.SESSION_MAP.get(sess_key, {}).get('label', sess_key)

    tr = traces.get(SUBJECT, {}).get(sess_key)
    if tr is None or tr.get('sfreq') is None:
        print(f'{sess_key}: no session trace — skipping')
        continue
    sfreq = float(tr['sfreq'])

    keep = [t for t in trials
            if (SOUND_SPLIT_INCLUDE_REJECTED or not t.get('rejected'))
            and t.get('epoch_anal') is not None and len(t['epoch_anal']) > 1
            and t.get('times_wide') is not None and len(t['times_wide']) > 1]
    keep.sort(key=lambda t: t['trial_index'])
    sound   = [t for t in keep if t.get('has_sound')]
    nosound = [t for t in keep if not t.get('has_sound')]

    N = len(nosound)
    if N == 0:
        print(f'{sess_key}: no no-sound trials — nothing to match against, skipping')
        continue
    if len(sound) < N:
        print(f'{sess_key}: only {len(sound)} sound trials (< N={N}) — skipping')
        continue

    # How many N-sized sound panels fit, capped at SOUND_MATCH_PANELS.
    n_panels = max(1, min(SOUND_MATCH_PANELS, len(sound) // N))
    # Evenly spaced block starts across the sorted sound list (early/mid/late).
    if n_panels == 1:
        starts = [0]
    else:
        starts = [int(round(o)) for o in np.linspace(0, len(sound) - N, n_panels)]
    sound_groups = [sound[s:s + N] for s in starts]

    rows = n_panels + 1
    fig, axes = plt.subplots(rows, 1, figsize=(14, 2.1 * rows),
                             sharex=True, sharey=True)
    axes = np.atleast_1d(axes)

    for k, grp in enumerate(sound_groups):
        a, b = grp[0]['trial_index'] + 1, grp[-1]['trial_index'] + 1
        _draw_run(axes[k], grp, sfreq,
                  f'Sound block {k+1} — {N} trials (idx {a}–{b})', mark_sound=True)
    _draw_run(axes[-1], nosound, sfreq,
              f'No-sound — all {N} trials', mark_sound=False)

    axes[-1].set_xlabel('Concatenated trial time (s) — each trial end-to-end, '
                        'grey lines separate trials', fontsize=9)
    handles = [Line2D([0], [0], color='k',     ls=':', lw=1.2, label='picture onset (t=0)'),
               Line2D([0], [0], color='green', ls=':', lw=1.2, label='sound onset (D110)'),
               Line2D([0], [0], color='black', lw=0.8, alpha=0.4, label='trial separator')]
    axes[0].legend(handles=handles, fontsize=8, loc='upper left', ncol=3, framealpha=0.85)

    fig.suptitle(f'{SUBJECT} · {sess_label} · sound vs no-sound '
                 f'(count-matched, N={N} trials/panel)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout(rect=(0, 0, 1, 0.97))

    if SAVE_FIGS:
        outdir = os.path.join(config.OUTPUT_DIR, 'glm_show', SUBJECT)
        os.makedirs(outdir, exist_ok=True)
        fig.savefig(os.path.join(outdir, f'{SUBJECT}_{sess_key}_sound_split.png'),
                    dpi=config.PLOT_DPI, bbox_inches='tight')
    plt.show()
    plt.close(fig)


### View 2d — Sound vs No-Sound response test (paired Wilcoxon)


In [ ]:
# response amp = ptp(response window) - ptp(baseline); paired Wilcoxon across subjects
def _trial_amp(t):
    ep = np.asarray(t['epoch_anal'], float)
    tw = np.asarray(t['times_wide'], float)
    base = ep[tw < 0]           # baseline window  [D105, code)
    resp = ep[tw >= 0]          # response window  [code, next code)
    if base.size < 2 or resp.size < 2:
        return None
    return float(np.ptp(resp) - np.ptp(base))   # baseline-corrected amplitude range

rows = []
for subj in sorted(trials_data):
    snd, nos = [], []
    for sess in trials_data[subj].values():
        for t in sess:
            if t.get('rejected') or t.get('epoch_anal') is None or t.get('times_wide') is None:
                continue
            v = _trial_amp(t)
            if v is None:
                continue
            (snd if t.get('has_sound') else nos).append(v)
    if snd and nos:
        rows.append((subj, len(snd), len(nos), float(np.mean(snd)), float(np.mean(nos))))

a = np.array([r[3] for r in rows])   # sound means
b = np.array([r[4] for r in rows])   # no-sound means
diffs = a - b
n = len(diffs)

w2, p2 = _stats.wilcoxon(a, b, alternative='two-sided')
wg, pg = _stats.wilcoxon(a, b, alternative='greater')
dz = float(diffs.mean() / diffs.std(ddof=1)) if n > 1 else float('nan')
npos = int((diffs > 0).sum())

print("Per-trial metric: ptp(response [code,next code)) - ptp(baseline [D105,code)), z units")
print(f"{'subj':6}{'n_snd':>6}{'n_nos':>6}{'sound':>9}{'no_snd':>9}{'diff':>9}")
for (subj, ns, nn, ms, mn), d in zip(rows, diffs):
    print(f"{subj:6}{ns:6d}{nn:6d}{ms:9.3f}{mn:9.3f}{d:+9.3f}")
print(f"\nN pairs (subjects) = {n}   |   sound>no-sound in {npos}/{n}   |   "
      f"median diff = {np.median(diffs):+.3f} z")
print(f"Wilcoxon two-sided:  W={w2:.1f}  p={p2:.4f}")
print(f"Wilcoxon one-sided (sound>no-sound):  W={wg:.1f}  p={pg:.4f}")
print(f"Cohen's dz = {dz:+.3f}")

# ── paired connecting-line plot ───────────────────────────────────────────────
C_NOS, C_SND = '#2980B9', '#C0392B'
fig, ax = plt.subplots(figsize=(6, 5))
for ms, mn in zip(a, b):
    ax.plot([0, 1], [mn, ms], color='#BDC3C7', lw=1.0, zorder=1)
ax.scatter(np.zeros(n), b, color=C_NOS, s=36, zorder=3, label='no-sound')
ax.scatter(np.ones(n),  a, color=C_SND, s=36, zorder=3, label='sound')
ax.plot([-0.15, 0.15], [np.median(b)]*2, color=C_NOS, lw=2.5, zorder=4)
ax.plot([0.85, 1.15],  [np.median(a)]*2, color=C_SND, lw=2.5, zorder=4)
ax.set_xticks([0, 1]); ax.set_xticklabels(['No-sound', 'Sound'])
ax.set_xlim(-0.4, 1.4)
ax.set_ylabel('Response amplitude range (z)\nptp(response) − ptp(baseline)', fontsize=9)
sig = 'n.s.' if p2 >= 0.05 else ('*' if p2 >= 0.01 else '**')
ax.set_title(f'Sound vs No-sound response  ·  paired Wilcoxon p={p2:.3f} ({sig})  ·  N={n}',
             fontsize=10, fontweight='bold')
ax.legend(fontsize=8, loc='upper right')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()

if SAVE_FIGS:
    outdir = os.path.join(config.OUTPUT_DIR, 'glm_show', '_group')
    os.makedirs(outdir, exist_ok=True)
    fig.savefig(os.path.join(outdir, 'sound_vs_nosound_wilcoxon.png'),
                dpi=config.PLOT_DPI, bbox_inches='tight')
plt.show()
plt.close(fig)


### View 3 — Trial Score Scatter (per subject)


In [ ]:
if SHOW_PER_SUBJECT_SCATTER:
    metric = GROUP_METRIC
    key = f'score_{metric}'

    for subj, sessions in sorted(trials_data.items()):
        fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
        fig.suptitle(f'{subj} – GLM {metric.upper()} Score by Trial Number', fontsize=13, fontweight='bold')

        for ax, sess_key in zip(axes, ['mor', 'eve']):
            trials = sessions.get(sess_key, [])
            if not trials:
                ax.set_title(f'{sess_key.upper()} – no data'); continue

            scores   = np.array([t.get(key) if t.get(key) is not None else np.nan for t in trials], dtype=float)
            labels   = np.array([t.get('label', 0) for t in trials])
            rejected = np.array([bool(t.get('rejected')) for t in trials])
            t_nums   = np.arange(1, len(trials) + 1)

            for cond, color, name in [(1, config.NEG_COLOR, 'Negative'), (2, config.NEU_COLOR, 'Neutral')]:
                mask = (~rejected) & (labels == cond) & np.isfinite(scores)
                if mask.sum() > 0:
                    ax.scatter(t_nums[mask], scores[mask], color=color,
                               s=60, edgecolors='white', linewidths=0.5, zorder=3,
                               label=f'{name} (n={mask.sum()})')
            n_rej = rejected.sum()
            if n_rej > 0:
                ax.scatter(t_nums[rejected], np.zeros(n_rej),
                           color='#95A5A6', marker='x', s=80, zorder=2,
                           label=f'Rejected (n={n_rej})')

            ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
            sess_label = config.SESSION_MAP.get(sess_key, {}).get('label', sess_key)
            ax.set_title(sess_label, fontsize=11, fontweight='bold')
            ax.set_xlabel('Trial number')
            if ax is axes[0]:
                ax.set_ylabel(f'GLM score ({metric.upper()}, beta weight)')
            ax.legend(fontsize=9, framealpha=0.9)
            ax.grid(True, linestyle='--', alpha=0.4)

        plt.tight_layout()
        plt.show()
        plt.close(fig)
else:
    print(f"SHOW_PER_SUBJECT_SCATTER is False — skipping (would show GROUP_METRIC="
          f"'{GROUP_METRIC}' scatter for all {len(trials_data)} subjects; set True in cell 1 to view).")

---
## Group-Level Views


### View 4 — Subject Average Timecourse


In [ ]:
# common time grid: trials are variable-length, so interpolate each onto one fixed grid
_all_trials = [t for sess in trials_data.values() for trials in sess.values()
               for t in trials if t.get("times_anal") is not None and len(t["times_anal"]) > 0]
T_MIN = max(t["times_anal"][0] for t in _all_trials)    # least-negative baseline extent
T_MAX = min(t["times_anal"][-1] for t in _all_trials)   # smallest response extent
GRID_DT = 1.0 / config.GLM_TARGET_SFREQ
TIMES_S = np.arange(T_MIN, T_MAX + GRID_DT, GRID_DT)
print(f"View 4 common time grid: {T_MIN:.2f}s to {T_MAX:.2f}s ({len(TIMES_S)} samples @ {config.GLM_TARGET_SFREQ}Hz)")

for subj, sessions in sorted(trials_data.items()):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    fig.suptitle(f"{subj} — Average Airflow Timecourse (GLM path, accepted trials)",
                 fontsize=13, fontweight="bold")
    has_data = False

    for ax, sess_key in zip(axes, ["eve", "mor"]):
        sess_label = config.SESSION_MAP.get(sess_key, {}).get("label", sess_key)
        trials = sessions.get(sess_key, [])

        if not trials:
            ax.set_title(f"{sess_label} — no data")
            continue

        for cond, color, name in [(1, config.NEG_COLOR, "Negative"), (2, config.NEU_COLOR, "Neutral")]:
            cond_trials = [
                t for t in trials
                if not t.get("rejected")
                and t.get("epoch_anal") is not None
                and t.get("baseline_mean") is not None
                and np.isfinite(t["baseline_mean"])
                and t.get("label") == cond
            ]
            if not cond_trials:
                continue
            has_data = True

            # Resample each trial onto TIMES_S using ITS OWN true time axis
            # (times_anal has t=0 at that trial's onset) -- NaN outside a
            # trial's own extent rather than pulling in the wrong time.
            arr = np.array([
                np.interp(TIMES_S, t["times_anal"], t["epoch_anal"] - t["baseline_mean"],
                          left=np.nan, right=np.nan)
                for t in cond_trials
            ])
            avg = np.nanmean(arr, axis=0)
            sem = (_stats.sem(arr, axis=0, nan_policy="omit") if len(cond_trials) > 1
                   else np.zeros_like(avg))

            ax.plot(TIMES_S, avg, color=color, linewidth=2.2,
                    label=f"{name} (n={len(cond_trials)})")
            if len(cond_trials) > 1:
                ax.fill_between(TIMES_S, avg - sem, avg + sem, color=color, alpha=0.20)

        ax.axvline(0, color="k", linestyle="--", linewidth=1.2, label="Picture onset")
        ax.axvspan(TIMES_S[0], 0.0, color="#3498DB", alpha=0.08, label="Baseline")
        ax.axvspan(0.0, TIMES_S[-1], color="green", alpha=0.08, label="Response window")
        ax.set_title(sess_label, fontsize=12, fontweight="bold")
        ax.set_xlabel("Time (s)", fontsize=11)
        if ax is axes[0]:
            ax.set_ylabel("Airflow (raw_z, baseline-corrected)", fontsize=11)
        ax.legend(fontsize=9, framealpha=0.9)
        ax.grid(True, linestyle="--", alpha=0.4)

    if has_data:
        plt.tight_layout()
        plt.show()
    plt.close(fig)

### View 5 — Group Average Timecourse (Neg / Neu split)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
fig.suptitle('Group Average Airflow Timecourse (GLM path, accepted trials)',
             fontsize=13, fontweight='bold')

for ax, sess_key in zip(axes, ['eve', 'mor']):
    sess_label = config.SESSION_MAP.get(sess_key, {}).get('label', sess_key)

    for cond, color, name in [(1, config.NEG_COLOR, 'Negative'), (2, config.NEU_COLOR, 'Neutral')]:
        subj_avgs = []
        n_subjs_g, n_trials_g = 0, 0

        for subj, sessions in trials_data.items():
            trials = sessions.get(sess_key, [])
            cond_trials = [
                t for t in trials
                if not t.get('rejected')
                and t.get('epoch_anal') is not None
                and t.get('baseline_mean') is not None
                and np.isfinite(t['baseline_mean'])
                and t.get('label') == cond
            ]
            if not cond_trials:
                continue
            arr = np.array([
                np.interp(TIMES_S, t['times_anal'], t['epoch_anal'] - t['baseline_mean'],
                          left=np.nan, right=np.nan)
                for t in cond_trials
            ])
            subj_avgs.append(np.nanmean(arr, axis=0))
            n_subjs_g += 1
            n_trials_g += len(cond_trials)

        if not subj_avgs:
            continue
        arr2 = np.array(subj_avgs)
        avg = np.nanmean(arr2, axis=0)
        sem = (_stats.sem(arr2, axis=0, nan_policy='omit') if len(subj_avgs) > 1
               else np.zeros_like(avg))

        ax.plot(TIMES_S, avg, color=color, linewidth=2.2,
                label=f'{name} (n subj={n_subjs_g}, n trials={n_trials_g})')
        ax.fill_between(TIMES_S, avg - sem, avg + sem, color=color, alpha=0.20)

    ax.axvline(0, color='k', linestyle='--', linewidth=1.2, label='Picture onset')
    ax.axvspan(TIMES_S[0], 0.0, color='#3498DB', alpha=0.08, label='Baseline')
    ax.axvspan(0.0, TIMES_S[-1], color='green', alpha=0.08, label='Response window')
    ax.set_title(sess_label, fontsize=12, fontweight='bold')
    ax.set_xlabel('Time (s)', fontsize=11)
    if ax is axes[0]:
        ax.set_ylabel('Airflow (raw_z, baseline-corrected, ±SEM)', fontsize=11)
        ax.legend(fontsize=9, framealpha=0.9, loc='upper left')
    ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()
plt.close(fig)

### View 6 — Group Overall Timecourse (Evening vs Morning)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for sess_key, sess_label, color in [('eve', 'Evening', config.EVE_COLOR),
                                     ('mor', 'Morning', config.MOR_COLOR)]:
    subj_avgs = []
    n_subjs_ov, n_trials_ov = 0, 0

    for subj, sessions in trials_data.items():
        trials = sessions.get(sess_key, [])
        cond_trials = [
            t for t in trials
            if not t.get('rejected')
            and t.get('epoch_anal') is not None
            and t.get('baseline_mean') is not None
            and np.isfinite(t['baseline_mean'])
        ]
        if not cond_trials:
            continue
        arr = np.array([
            np.interp(TIMES_S, t['times_anal'], t['epoch_anal'] - t['baseline_mean'],
                      left=np.nan, right=np.nan)
            for t in cond_trials
        ])
        subj_avgs.append(np.nanmean(arr, axis=0))
        n_subjs_ov += 1
        n_trials_ov += len(cond_trials)

    if not subj_avgs:
        continue
    arr2 = np.array(subj_avgs)
    avg = np.nanmean(arr2, axis=0)
    sem = (_stats.sem(arr2, axis=0, nan_policy='omit') if len(subj_avgs) > 1
           else np.zeros_like(avg))

    ax.plot(TIMES_S, avg, color=color, linewidth=2.5,
            label=f'{sess_label} (n subj={n_subjs_ov}, n trials={n_trials_ov})')
    ax.fill_between(TIMES_S, avg - sem, avg + sem, color=color, alpha=0.20)

ax.axvline(0, color='k', linestyle='--', linewidth=1.2, label='Picture onset')
ax.axvspan(TIMES_S[0], 0.0, color='#3498DB', alpha=0.08, label='Baseline')
ax.axvspan(0.0, TIMES_S[-1], color='green', alpha=0.08, label='Response window')
ax.set_xlabel('Time (s)', fontsize=11)
ax.set_ylabel('Airflow (raw_z, baseline-corrected, ±SEM)', fontsize=11)
ax.set_title('Overall Average Airflow Timecourse (GLM path): Evening vs Morning\n'
             '(all accepted trials, ±SEM shaded)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, framealpha=0.9, loc='upper left')
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()
plt.close(fig)

## ⭐ Confirmatory endpoint — Evening > Morning (pre-registered)


In [ ]:
# CONFIRMATORY ENDPOINT -- the single pre-registered Evening > Morning test
PRIMARY = config.GLM_PRIMARY_METRIC.lower()      # 'ra' | 'rfr' | 'rp' | 'composite'
PRIMARY_SOUND_ONLY = False                        # True = sound trials only

def _session_metric_mean(trials, metric):
    key = f'score_{metric}'
    vals = [t[key] for t in trials
            if not t['rejected'] and t.get(key) is not None and np.isfinite(t[key])
            and (not PRIMARY_SOUND_ONLY or t.get('has_sound'))]
    return float(np.mean(vals)) if vals else np.nan

def _primary_eve_mor():
    """Per-subject (Evening, Morning) summary of the primary endpoint over accepted
    trials (valence collapsed). 'composite' = mean of the three per-metric means,
    each z-scored across all (subject, session) cells so RP/RA/RFR share one scale."""
    if PRIMARY in ('rp', 'ra', 'rfr'):
        return {subj: (_session_metric_mean(sess.get('eve', []), PRIMARY),
                       _session_metric_mean(sess.get('mor', []), PRIMARY))
                for subj, sess in trials_data.items()}
    raw = {(subj, sk): {mt: _session_metric_mean(sess.get(sk, []), mt)
                        for mt in ('rp', 'ra', 'rfr')}
           for subj, sess in trials_data.items() for sk in ('eve', 'mor')}
    zst = {}
    for mt in ('rp', 'ra', 'rfr'):
        col = np.array([v[mt] for v in raw.values()], float)
        mu, sd = np.nanmean(col), np.nanstd(col)
        zst[mt] = (mu, sd if sd > 0 else 1.0)
    rows = {}
    for subj, sess in trials_data.items():
        comp = {}
        for sk in ('eve', 'mor'):
            zs = [(raw[(subj, sk)][mt] - zst[mt][0]) / zst[mt][1] for mt in ('rp', 'ra', 'rfr')]
            comp[sk] = float(np.mean(zs)) if np.all(np.isfinite(zs)) else np.nan
        rows[subj] = (comp['eve'], comp['mor'])
    return rows

_pairs  = _primary_eve_mor()
_paired = [(s, e, m) for s, (e, m) in _pairs.items() if np.isfinite(e) and np.isfinite(m)]
_eve = [e for _, e, m in _paired]; _mor = [m for _, e, m in _paired]; _N = len(_paired)

_bar = '═' * 72
print(_bar)
print(f'  CONFIRMATORY ENDPOINT  --  primary metric = {config.GLM_PRIMARY_METRIC}'
      f'{"  (sound-only)" if PRIMARY_SOUND_ONLY else ""}')
print(_bar)
print(f'  aggregation : per-subject mean of score_{PRIMARY} over accepted trials,')
print(f'                Evening vs Morning (valence collapsed)')
print(f'  estimation  : {getattr(config, "GLM_ESTIMATION", "per_trial")}   (per_trial = one fit per trial; pooled_session = PsPM pspm_glm.m,')
print(f'                one session-wide fit, one beta per condition -- see GLM_METHOD_FOUNDATIONS.md 5)')
print(f'  N (subjects with both sessions) : {_N}')
if _N >= 2:
    _n_gt = sum(e > m for _, e, m in _paired)
    print(f'  median   Evening = {np.median(_eve):+.5f}    Morning = {np.median(_mor):+.5f}')
    print(f'  Evening > Morning in {_n_gt} / {_N} subjects')
    try:
        _w,  _p2 = _stats.wilcoxon(_eve, _mor)
        _w1, _p1 = _stats.wilcoxon(_eve, _mor, alternative='greater')
        print(f'  Wilcoxon signed-rank   W = {_w:.1f}')
        print(f'     two-sided p = {_p2:.4f}   (no directional claim)')
        print(f'     one-sided p = {_p1:.4f}   (H1: Evening > Morning)')
        print('  The one-sided p is legitimate ONLY because the direction was fixed a')
        print('  priori; report it AS one-sided, alongside the two-sided p, not instead of it.')
    except Exception as _ex:
        print(f'  [!] Wilcoxon failed: {_ex}')
else:
    print('  [!] fewer than 2 paired subjects -- cannot run the test')
print(_bar)

### View 7 — Group 4-Condition Boxplot


In [ ]:
def _sig_label(pval):
    return "***" if pval < 0.001 else ("**" if pval < 0.01 else ("*" if pval < 0.05 else "ns"))

def _draw_stat_bracket(ax, x1, x2, y_top, pval, test_label, fontsize=10):
    h = (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.015
    ax.plot([x1, x1, x2, x2], [y_top, y_top + h, y_top + h, y_top], lw=1.2, color='black')
    ax.text((x1 + x2) / 2, y_top + h * 1.5, f'{_sig_label(pval)}  {test_label} p={pval:.3f}',
            ha='center', va='bottom', fontsize=fontsize, fontweight='bold')

def _paired_test(a, b):
    try:
        stat, p = _stats.wilcoxon(a, b); return stat, p, 'Wilcoxon'
    except Exception:
        stat, p = _stats.ttest_rel(a, b); return stat, p, 'Paired t'

def _subject_condition_means(metric):
    key = f'score_{metric}'
    out = {}
    for subj, sessions in trials_data.items():
        out[subj] = {}
        for sk in ('eve', 'mor'):
            for label in (1, 2):
                vals = [t[key] for t in sessions.get(sk, [])
                        if not t['rejected'] and t['condition'] == label
                        and t.get(key) is not None and np.isfinite(t[key])]
                if vals:
                    out[subj][(sk, label)] = float(np.mean(vals))
    return out

def _metric_method_note(metric):
    """One-line description of what score_{metric} actually measures, for
    display under any plot titled '... by condition' -- see fit_trial_glm's
    docstring in airflow_glm.py: score = beta_1, the CRF-template-matching
    coefficient from a regression over the trial's full baseline+response
    window, not a peak read-off."""
    tau, sigma = config.GLM_RF_PARAMS[metric.upper()]
    deriv = ' + dCRF/dt' if config.GLM_USE_DERIVATIVE.get(metric.upper()) else ''
    return f'score = β₁ from GLM fit vs Gaussian CRF (τ={tau:.2f}s, σ={sigma:.2f}s){deriv}'

_condition_display = {
    "label": {1: "Negative", 2: "Neutral"},
    "has_sound": {1: "Sound", 2: "No-sound"},
}
_cond_name = _condition_display[getattr(config, "GLM_CONDITION_FIELD", "label")]

cond_names = {('eve', 1): f'Eve-{_cond_name[1]}', ('eve', 2): f'Eve-{_cond_name[2]}',
              ('mor', 1): f'Mor-{_cond_name[1]}', ('mor', 2): f'Mor-{_cond_name[2]}'}
cond_keys_glm = [('eve', 1, f'Eve-{_cond_name[1]}', config.NEG_COLOR,    '#E8A99A'),
                 ('eve', 2, f'Eve-{_cond_name[2]}', config.NEU_COLOR,    '#9AC4E8'),
                 ('mor', 1, f'Mor-{_cond_name[1]}', '#922B21',            '#E8A99A'),
                 ('mor', 2, f'Mor-{_cond_name[2]}', '#1A5276',            '#9AC4E8')]

for metric in ('rp', 'ra', 'rfr'):
    print(f'\n[GLM stats] metric={metric.upper()}')
    subj_means = _subject_condition_means(metric)

    for cond, name in cond_names.items():
        vals = [m[cond] for m in subj_means.values() if cond in m]
        if len(vals) > 1:
            t_stat, p = _stats.ttest_1samp(vals, 0.0)
            print(f'  {name} vs 0: t({len(vals)-1})={t_stat:.3f}, p={p:.4f}, n={len(vals)}')

    for sk, sk_label in (('eve', 'Evening'), ('mor', 'Morning')):
        subjs = [s for s, m in subj_means.items() if (sk, 1) in m and (sk, 2) in m]
        if len(subjs) > 1:
            a = [subj_means[s][(sk, 1)] for s in subjs]; b = [subj_means[s][(sk, 2)] for s in subjs]
            stat, p, lbl = _paired_test(a, b)
            print(f'  {sk_label} {_cond_name[1]} vs {_cond_name[2]}: {lbl} stat={stat:.3f}, p={p:.4f}, n={len(subjs)}')

    for label, label_name in _cond_name.items():
        subjs = [s for s, m in subj_means.items() if ('eve', label) in m and ('mor', label) in m]
        if len(subjs) > 1:
            a = [subj_means[s][('eve', label)] for s in subjs]; b = [subj_means[s][('mor', label)] for s in subjs]
            stat, p, lbl = _paired_test(a, b)
            print(f'  {label_name} Eve vs Mor: {lbl} stat={stat:.3f}, p={p:.4f}, n={len(subjs)}')

    valid_subjs = [s for s in subj_means if all((sk, c) in subj_means[s] for sk, c, *_ in cond_keys_glm)]
    if len(valid_subjs) < 2:
        print(f'  [plot] {metric}: fewer than 2 subjects with all 4 conditions — skipping boxplot')
        continue

    data_cols = [[subj_means[s][(sk, c)] for s in valid_subjs] for sk, c, *_ in cond_keys_glm]
    fig, ax = plt.subplots(figsize=(10, 6))
    positions = [1, 2, 3, 4]
    bp = ax.boxplot(data_cols, positions=positions, widths=0.45, patch_artist=True, showfliers=False)
    for patch, (_, _, _, _, col) in zip(bp['boxes'], cond_keys_glm):
        patch.set_facecolor(col); patch.set_alpha(0.7); patch.set_edgecolor('gray')
    for median in bp['medians']:
        median.set(color='black', linewidth=2)

    for s in valid_subjs:
        j = np.random.uniform(-0.07, 0.07)
        v_en, v_eu = subj_means[s][('eve', 1)], subj_means[s][('eve', 2)]
        lc = config.UP_COLOR if v_eu >= v_en else config.DOWN_COLOR
        ax.plot([1 + j, 2 + j], [v_en, v_eu], color=lc, alpha=0.45, linewidth=1.2)
        ax.scatter([1 + j, 2 + j], [v_en, v_eu], color=[config.NEG_COLOR, config.NEU_COLOR],
                   edgecolors='k', s=45, zorder=4, linewidths=0.5)
        v_mn, v_mu = subj_means[s][('mor', 1)], subj_means[s][('mor', 2)]
        lc = config.UP_COLOR if v_mu >= v_mn else config.DOWN_COLOR
        ax.plot([3 + j, 4 + j], [v_mn, v_mu], color=lc, alpha=0.45, linewidth=1.2)
        ax.scatter([3 + j, 4 + j], [v_mn, v_mu], color=[config.NEG_COLOR, config.NEU_COLOR],
                   edgecolors='k', s=45, zorder=4, linewidths=0.5)

    eve_neg, eve_neu, mor_neg, mor_neu = data_cols
    _, p_eve, tl_eve = _paired_test(eve_neg, eve_neu)
    _, p_mor, tl_mor = _paired_test(mor_neg, mor_neu)
    all_vals = eve_neg + eve_neu + mor_neg + mor_neu
    y_span = max(all_vals) - min(all_vals) if max(all_vals) != min(all_vals) else 1.0
    ax.set_ylim(min(all_vals) - 0.15 * y_span, max(all_vals) + 0.35 * y_span)
    _draw_stat_bracket(ax, 1, 2, max(max(eve_neg), max(eve_neu)) + 0.05 * y_span, p_eve, tl_eve)
    _draw_stat_bracket(ax, 3, 4, max(max(mor_neg), max(mor_neu)) + 0.05 * y_span, p_mor, tl_mor)

    ax.set_xticks(positions); ax.set_xticklabels([c[2] for c in cond_keys_glm])
    ax.set_ylabel(f'GLM score ({metric.upper()}, beta weight)')
    ax.set_title(f'Airflow GLM — {metric.upper()} by condition (n={len(valid_subjs)} subjects)', pad=22)
    ax.text(0.5, 1.0, _metric_method_note(metric), transform=ax.transAxes,
            ha='center', va='bottom', fontsize=8, color='dimgray', style='italic')
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ax.grid(True, axis='y', linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()
    plt.close(fig)

### View 8 — Session Comparison (Eve vs Mor, by condition)


In [ ]:
if getattr(config, "GLM_CONDITION_FIELD", "label") != "label":
    print(f"[View 8] skipped — hardcoded for condition=label (Negative/Neutral); "
          f"current GLM_CONDITION_FIELD={getattr(config, 'GLM_CONDITION_FIELD', 'label')!r}")
else:
    cond_keys7 = [('eve', 1, 'Eve-Neg', config.NEG_COLOR),
                  ('mor', 1, 'Mor-Neg', '#922B21'),
                  ('eve', 2, 'Eve-Neu', config.NEU_COLOR),
                  ('mor', 2, 'Mor-Neu', '#1A5276')]
    pos7 = [1, 2, 3.6, 4.6]

    for metric in ('rp', 'ra', 'rfr'):
        subj_means = _subject_condition_means(metric)
        valid7 = [s for s in subj_means if all((sk, c) in subj_means[s] for sk, c, *_ in cond_keys7)]
        if len(valid7) < 2:
            print(f'[View 8] {metric}: fewer than 2 subjects with all 4 conditions — skipping')
            continue
        data7 = [[subj_means[s][(sk, c)] for s in valid7] for sk, c, *_ in cond_keys7]

        fig, ax = plt.subplots(figsize=(10, 6))
        bp = ax.boxplot(data7, positions=pos7, widths=0.42, patch_artist=True, showfliers=False)
        for patch, (_, _, _, col) in zip(bp['boxes'], cond_keys7):
            patch.set_facecolor(col); patch.set_alpha(0.65); patch.set_edgecolor('gray')
        for median in bp['medians']:
            median.set(color='black', linewidth=2)

        for s in valid7:
            j = np.random.uniform(-0.07, 0.07)
            v_en, v_mn = subj_means[s][('eve', 1)], subj_means[s][('mor', 1)]
            lc = config.UP_COLOR if v_mn >= v_en else config.DOWN_COLOR
            ax.plot([1 + j, 2 + j], [v_en, v_mn], color=lc, alpha=0.55, linewidth=1.4)
            ax.scatter([1 + j, 2 + j], [v_en, v_mn], color=[cond_keys7[0][3], cond_keys7[1][3]],
                       edgecolors='k', s=45, zorder=4, linewidths=0.5)
            v_eu, v_mu = subj_means[s][('eve', 2)], subj_means[s][('mor', 2)]
            lc = config.UP_COLOR if v_mu >= v_eu else config.DOWN_COLOR
            ax.plot([3.6 + j, 4.6 + j], [v_eu, v_mu], color=lc, alpha=0.55, linewidth=1.4)
            ax.scatter([3.6 + j, 4.6 + j], [v_eu, v_mu], color=[cond_keys7[2][3], cond_keys7[3][3]],
                       edgecolors='k', s=45, zorder=4, linewidths=0.5)

        neg_eve7, neg_mor7, neu_eve7, neu_mor7 = data7
        _, p_neg7, tl_neg7 = _paired_test(neg_eve7, neg_mor7)
        _, p_neu7, tl_neu7 = _paired_test(neu_eve7, neu_mor7)
        all_vals7 = neg_eve7 + neg_mor7 + neu_eve7 + neu_mor7
        y_span7 = max(all_vals7) - min(all_vals7) if max(all_vals7) != min(all_vals7) else 1.0
        ax.set_ylim(min(all_vals7) - 0.15 * y_span7, max(all_vals7) + 0.35 * y_span7)
        _draw_stat_bracket(ax, 1, 2, max(max(neg_eve7), max(neg_mor7)) + 0.05 * y_span7, p_neg7, tl_neg7)
        _draw_stat_bracket(ax, 3.6, 4.6, max(max(neu_eve7), max(neu_mor7)) + 0.05 * y_span7, p_neu7, tl_neu7)

        ax.set_xticks(pos7)
        ax.set_xticklabels([c[2] for c in cond_keys7])
        ax.set_ylabel(f'GLM score ({metric.upper()}, beta weight)')
        ax.set_title(f'Airflow GLM — {metric.upper()}: Evening vs Morning by condition (n={len(valid7)} subjects)',
                     pad=22)
        ax.text(0.5, 1.0, _metric_method_note(metric), transform=ax.transAxes,
                ha='center', va='bottom', fontsize=8, color='dimgray', style='italic')
        ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
        ax.axvline(2.8, color='lightgray', linestyle='--', linewidth=1)
        ax.grid(True, axis='y', linestyle='--', alpha=0.3)
        plt.tight_layout()
        plt.show()
        plt.close(fig)

### View 9 — Neg/Neu Ratio (Evening vs Morning)


In [ ]:
if getattr(config, "GLM_CONDITION_FIELD", "label") != "label":
    print(f"[View 9] skipped — hardcoded for condition=label (Negative/Neutral); "
          f"current GLM_CONDITION_FIELD={getattr(config, 'GLM_CONDITION_FIELD', 'label')!r}")
else:
    for metric in ('rp', 'ra', 'rfr'):
        subj_means = _subject_condition_means(metric)
        eve_rat8, mor_rat8, valid8 = [], [], []

        for subj, m in subj_means.items():
            rat = {}
            for sk in ('eve', 'mor'):
                if (sk, 1) in m and (sk, 2) in m and m[(sk, 2)] != 0:
                    rat[sk] = m[(sk, 1)] / m[(sk, 2)]
            if 'eve' in rat and 'mor' in rat:
                eve_rat8.append(rat['eve']); mor_rat8.append(rat['mor']); valid8.append(subj)

        n8m = len(valid8)
        if n8m < 2:
            print(f'[View 9] {metric}: fewer than 2 subjects with both ratios — skipping')
            continue
        _, pval8, tl8 = _paired_test(eve_rat8, mor_rat8)

        fig, ax = plt.subplots(figsize=(6, 6))
        bp = ax.boxplot([eve_rat8, mor_rat8], positions=[1, 2], widths=0.4,
                        patch_artist=True, showfliers=False)
        for patch, color in zip(bp['boxes'], [config.EVE_COLOR, config.MOR_COLOR]):
            patch.set_facecolor(color); patch.set_alpha(0.6); patch.set_edgecolor('gray')
        for median in bp['medians']:
            median.set(color='black', linewidth=2)

        for ev, mo in zip(eve_rat8, mor_rat8):
            j = np.random.uniform(-0.05, 0.05)
            lc = config.UP_COLOR if mo >= ev else config.DOWN_COLOR
            ax.plot([1 + j, 2 + j], [ev, mo], color=lc, alpha=0.55, linewidth=1.2)
            ax.scatter(1 + j, ev, color=config.EVE_COLOR, edgecolors='k', s=55, zorder=3, linewidths=0.5)
            ax.scatter(2 + j, mo, color=config.MOR_COLOR, edgecolors='k', s=55, zorder=3, linewidths=0.5)

        y_top8m = max(max(eve_rat8), max(mor_rat8))
        y_span8m = y_top8m - min(min(eve_rat8), min(mor_rat8))
        y_span8m = y_span8m if y_span8m != 0 else 1.0
        ax.set_ylim(top=y_top8m + 0.30 * y_span8m)
        _draw_stat_bracket(ax, 1, 2, y_top8m + 0.06 * y_span8m, pval8, tl8)
        ax.set_xticks([1, 2])
        ax.set_xticklabels([f'Evening\n(n={n8m})', f'Morning\n(n={n8m})'], fontsize=11)
        ax.set_ylabel(f'Neg / Neu GLM Score Ratio ({metric.upper()})', fontsize=11)
        ax.set_title(f'Airflow GLM — {metric.upper()}: Negative-to-Neutral Ratio, Evening vs Morning\n'
                     f'(N={n8m} subjects)', fontsize=12, fontweight='bold')
        ax.axhline(1, color='gray', linestyle=':', linewidth=1.2)
        ax.grid(True, axis='y', linestyle='--', alpha=0.4)
        plt.tight_layout()
        plt.show()
        plt.close(fig)

### View 10 — Evening/Morning Ratio (by condition)


In [ ]:
if getattr(config, "GLM_CONDITION_FIELD", "label") != "label":
    print(f"[View 10] skipped — hardcoded for condition=label (Negative/Neutral); "
          f"current GLM_CONDITION_FIELD={getattr(config, 'GLM_CONDITION_FIELD', 'label')!r}")
else:
    for metric in ('rp', 'ra', 'rfr'):
        subj_means = _subject_condition_means(metric)
        neg_rat9, neu_rat9, valid9m = [], [], []

        for subj, m in subj_means.items():
            if all(k in m for k in [('eve', 1), ('mor', 1), ('eve', 2), ('mor', 2)]):
                if m[('mor', 1)] != 0 and m[('mor', 2)] != 0:
                    neg_rat9.append(m[('eve', 1)] / m[('mor', 1)])
                    neu_rat9.append(m[('eve', 2)] / m[('mor', 2)])
                    valid9m.append(subj)

        n9m = len(valid9m)
        if n9m < 2:
            print(f'[View 10] {metric}: fewer than 2 subjects with all 4 conditions — skipping')
            continue
        _, pval9, tl9 = _paired_test(neg_rat9, neu_rat9)

        fig, ax = plt.subplots(figsize=(6, 6))
        bp = ax.boxplot([neg_rat9, neu_rat9], positions=[1, 2], widths=0.4,
                        patch_artist=True, showfliers=False)
        for patch, color in zip(bp['boxes'], [config.NEG_COLOR, config.NEU_COLOR]):
            patch.set_facecolor(color); patch.set_alpha(0.55); patch.set_edgecolor('gray')
        for median in bp['medians']:
            median.set(color='black', linewidth=2)

        for neg_r, neu_r in zip(neg_rat9, neu_rat9):
            j = np.random.uniform(-0.05, 0.05)
            lc = config.UP_COLOR if neu_r >= neg_r else config.DOWN_COLOR
            ax.plot([1 + j, 2 + j], [neg_r, neu_r], color=lc, alpha=0.6, linewidth=1.4)
            ax.scatter(1 + j, neg_r, color=config.NEG_COLOR, edgecolors='k', s=55, zorder=3, linewidths=0.5)
            ax.scatter(2 + j, neu_r, color=config.NEU_COLOR, edgecolors='k', s=55, zorder=3, linewidths=0.5)

        y_top9m = max(max(neg_rat9), max(neu_rat9))
        y_span9m = y_top9m - min(min(neg_rat9), min(neu_rat9))
        y_span9m = y_span9m if y_span9m != 0 else 1.0
        ax.set_ylim(top=y_top9m + 0.30 * y_span9m)
        _draw_stat_bracket(ax, 1, 2, y_top9m + 0.06 * y_span9m, pval9, tl9)
        ax.axhline(1, color='gray', linestyle=':', linewidth=1.2)
        ax.set_xticks([1, 2])
        ax.set_xticklabels([f'Negative\n(n={n9m})', f'Neutral\n(n={n9m})'], fontsize=11)
        ax.set_ylabel(f'Evening / Morning GLM Score Ratio ({metric.upper()})', fontsize=11)
        ax.set_title(f'Airflow GLM — {metric.upper()}: Evening-to-Morning Ratio, Neg vs Neu (N={n9m} subjects)',
                     fontsize=12, fontweight='bold')
        ax.grid(True, axis='y', linestyle='--', alpha=0.4)
        plt.tight_layout()
        plt.show()
        plt.close(fig)

### View 11 — Overall Evening vs Morning


In [ ]:
for metric in ('rp', 'ra', 'rfr'):
    key = f'score_{metric}'
    eve_ov10, mor_ov10, valid_ov10 = [], [], []

    for subj, sessions in sorted(trials_data.items()):
        vals = {}
        for sk in ('eve', 'mor'):
            scores = [t[key] for t in sessions.get(sk, [])
                      if not t['rejected'] and t.get(key) is not None and np.isfinite(t[key])]
            if scores:
                vals[sk] = np.mean(scores)
        if 'eve' in vals and 'mor' in vals:
            eve_ov10.append(vals['eve']); mor_ov10.append(vals['mor']); valid_ov10.append(subj)

    n_ov10 = len(valid_ov10)
    if n_ov10 < 2:
        print(f'[View 11] {metric}: fewer than 2 subjects with both sessions — skipping')
        continue
    _, pval_ov10, tl_ov10 = _paired_test(eve_ov10, mor_ov10)

    nt_eve10 = sum(sum(1 for t in trials_data[s].get('eve', []) if not t['rejected']) for s in valid_ov10)
    nt_mor10 = sum(sum(1 for t in trials_data[s].get('mor', []) if not t['rejected']) for s in valid_ov10)

    fig, ax = plt.subplots(figsize=(6, 6))
    bp = ax.boxplot([eve_ov10, mor_ov10], positions=[1, 2], widths=0.4,
                    patch_artist=True, showfliers=False)
    for patch, color in zip(bp['boxes'], [config.EVE_COLOR, config.MOR_COLOR]):
        patch.set_facecolor(color); patch.set_alpha(0.6); patch.set_edgecolor('gray')
    for median in bp['medians']:
        median.set(color='black', linewidth=2)

    for ev, mo in zip(eve_ov10, mor_ov10):
        j = np.random.uniform(-0.05, 0.05)
        lc = config.UP_COLOR if mo >= ev else config.DOWN_COLOR
        ax.plot([1 + j, 2 + j], [ev, mo], color=lc, alpha=0.6, linewidth=1.4)
        ax.scatter(1 + j, ev, color=config.EVE_COLOR, edgecolors='k', s=55, zorder=3, linewidths=0.5)
        ax.scatter(2 + j, mo, color=config.MOR_COLOR, edgecolors='k', s=55, zorder=3, linewidths=0.5)

    y_top_ov10 = max(max(eve_ov10), max(mor_ov10))
    y_span_ov10 = y_top_ov10 - min(min(eve_ov10), min(mor_ov10))
    y_span_ov10 = y_span_ov10 if y_span_ov10 != 0 else 1.0
    ax.set_ylim(top=y_top_ov10 + 0.35 * y_span_ov10)
    _draw_stat_bracket(ax, 1, 2, y_top_ov10 + 0.05 * y_span_ov10, pval_ov10, tl_ov10)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(
        [f'Evening\n(n subj={n_ov10}, n trials={nt_eve10})',
         f'Morning\n(n subj={n_ov10}, n trials={nt_mor10})'], fontsize=10)
    ax.set_ylabel(f'GLM score ({metric.upper()}, beta weight)', fontsize=11)
    ax.set_title(f'Airflow GLM — {metric.upper()}: Overall Evening vs Morning\n'
                 f'(N={n_ov10} subjects, all accepted trials, green=↑ red=↓)',
                 fontsize=12, fontweight='bold')
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ax.grid(True, axis='y', linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()
    plt.close(fig)

### View 12 — Overall Eve/Mor Ratio


In [ ]:
for metric in ('rp', 'ra', 'rfr'):
    key = f'score_{metric}'
    all_rat11, valid11 = [], []

    for subj, sessions in sorted(trials_data.items()):
        vals = {}
        for sk in ('eve', 'mor'):
            scores = [t[key] for t in sessions.get(sk, [])
                      if not t['rejected'] and t.get(key) is not None and np.isfinite(t[key])]
            if scores:
                vals[sk] = np.mean(scores)
        if 'eve' in vals and 'mor' in vals and vals['mor'] != 0:
            all_rat11.append(vals['eve'] / vals['mor'])
            valid11.append(subj)

    n11 = len(valid11)
    if n11 < 2:
        print(f'[View 12] {metric}: fewer than 2 subjects with a valid ratio — skipping')
        continue
    try:
        _, pval11 = _stats.wilcoxon([r - 1 for r in all_rat11]); tl11 = 'Wilcoxon (vs 1)'
    except Exception:
        _, pval11 = _stats.ttest_1samp(all_rat11, 1); tl11 = 'One-sample t (vs 1)'

    fig, ax = plt.subplots(figsize=(5, 6))
    bp = ax.boxplot(all_rat11, positions=[1], widths=0.35, patch_artist=True, showfliers=False)
    bp['boxes'][0].set_facecolor('#95A5A6'); bp['boxes'][0].set_alpha(0.55)
    bp['boxes'][0].set_edgecolor('gray')
    bp['medians'][0].set(color='black', linewidth=2)

    for r in all_rat11:
        j = np.random.uniform(-0.08, 0.08)
        ax.scatter(1 + j, r, color='#7F8C8D', edgecolors='k', s=60, zorder=3, linewidths=0.5)

    ax.axhline(1, color='gray', linestyle=':', linewidth=1.5)
    y_top11 = max(all_rat11) * (1.08 if max(all_rat11) > 0 else 0.92)
    ax.set_ylim(top=y_top11 * (1.20 if y_top11 > 0 else 0.80))
    sig11 = _sig_label(pval11)
    ax.text(1, y_top11 * (1.05 if y_top11 > 0 else 0.95),
            f'{sig11}\n{tl11}\np={pval11:.3f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_xticks([1])
    ax.set_xticklabels([f'Eve / Mor\n(n={n11})'], fontsize=11)
    ax.set_ylabel(f'Evening / Morning GLM Score Ratio ({metric.upper()})', fontsize=11)
    ax.set_title(f'Airflow GLM — {metric.upper()}: Overall Eve/Mor Ratio (all accepted trials)\n'
                 'Tested against ratio=1 (ratio > 1 = Eve > Mor)',
                 fontsize=11, fontweight='bold')
    ax.grid(True, axis='y', linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()
    plt.close(fig)

## Archived — Raw MFF view (not needed for now)


### View 0 — Raw MFF (no filtering at all)


In [ ]:

for sk in sess_keys:
    trials = trials_data[SUBJECT].get(sk, [])
    n_rej  = sum(1 for t in trials if t.get('rejected'))
    print(f"{SUBJECT} {config.SESSION_MAP.get(sk,{}).get('label', sk)}: "
          f"{len(trials)} trials, {n_rej} rejected")

complete_subjs, incomplete_subjs = [], []
for subj, sessions in sorted(trials_data.items()):
    has_eve = bool(sessions.get('eve'))
    has_mor = bool(sessions.get('mor'))
    (complete_subjs if (has_eve and has_mor) else incomplete_subjs).append(
        (subj, has_eve, has_mor))

print(f"\n{len(complete_subjs)}/{len(trials_data)} subjects have both Eve and Mor "
      f"— only these are used in Eve-vs-Mor comparisons below.")
if incomplete_subjs:
    print("Excluded from Eve-vs-Mor comparisons (missing a session):")
    for subj, has_eve, has_mor in incomplete_subjs:
        if not has_eve and not has_mor:
            missing = '(no trials at all)'
        elif not has_eve:
            missing = 'Eve'
        else:
            missing = 'Mor'
        print(f"  {subj}: missing {missing}")